# Hidden Norm Plots

Per-token activation / gradient norm trajectories from `GradientBiasMonitor`
dumps. The reader (`read_norms.read_hidden`) returns dense arrays plus a
`global_steps` vector that may be **sparse** (e.g. `[0, 20, 40, ...]` when
`--monitor-record-every-k-steps > 1`).

- **X-axis**: relative token position (`-1` = last token, rightmost).
- **One unified plotting function** `plot_norms(...)` drives everything;
  see its docstring for the four-tuple `select` API.


In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, os.path.abspath('.'))

import itertools
from dataclasses import dataclass
from typing import List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.colors as pc

from read_norms import read_hidden


@dataclass
class View:
    """Wraps one norm array (act or grad) with its step / position metadata.

    arr           : (S, A, B, L, T) float32
                    -- (global_step, accum_step, batch_idx, layer, token_pos)
    global_steps  : (S,) int   -- the *real* (possibly sparse) global_step ids;
                                   axis 0 of `arr` indexes into this vector.
    rel_positions : (T,) int   -- x-axis values; `-1` is the last token.
    """
    arr: np.ndarray
    global_steps: np.ndarray
    rel_positions: np.ndarray


def _smooth(norms, window_avg):
    if window_avg <= 1:
        return norms
    return (pd.Series(norms)
              .rolling(window_avg, center=True, min_periods=1)
              .mean()
              .to_numpy(dtype=np.float32))


def _global_padded(arrs, pad_frac=0.05):
    lo = min(float(a.min()) for a in arrs)
    hi = max(float(a.max()) for a in arrs)
    pad = (hi - lo) * pad_frac if hi > lo else abs(lo) * pad_frac or 0.01
    return lo - pad, hi + pad


def _aligned_range(v, lo, hi, frac):
    span = max((v - lo) / max(frac, 1e-9), (hi - v) / max(1 - frac, 1e-9)) * 1.05
    return v - frac * span, v + (1 - frac) * span


def _tickvals(pos_min):
    step = max(1, 2 ** int(round(np.log2(abs(pos_min) / 8))))
    start = (pos_min // step) * step
    return [v for v in range(start, 0, step) if v != -1] + [-1]


## The single plotting function

```python
plot_norms(v_act, v_grad, select,
           collapse=None, normalize=False, window_avg=1, title='')
```

### Dimensions

The on-disk arrays are 5-D `(global_step, accum_step, batch_idx, layer, token_pos)`:

| dim     | meaning                                                                                |
|---------|----------------------------------------------------------------------------------------|
| `step`  | **global training step** — index into `view.global_steps` (sparse-safe; legend prints the real id) |
| `accum` | **gradient-accumulation micro-step** within one optimizer step                         |
| `batch` | **per-DDP-rank sample index**, flattened across `rank × device_batch_size`             |
| `layer` | transformer **block** index (0 = first block)                                          |
| token_pos | always the X-axis; never selected                                                    |

### `select` — 4-tuple over `(step, accum, batch, layer)`

| entry                  | meaning                                  |
|------------------------|------------------------------------------|
| `int`                  | fix that dim to a single index           |
| `list[int]`            | sweep over the listed indices            |
| `slice` / `np.s_[a:b]` | sweep over a half-open index range       |
| `None`                 | sweep over **all** values on that axis   |

By default each *swept* dim contributes one curve per index (cross-product when several dims sweep at once).

### `collapse` — average instead of sweep

`collapse` is `None`, a single dim name, or a list of names from `{'step', 'accum', 'batch', 'layer'}`. For every dim listed there, the indices selected by `select` are **averaged** instead of producing separate curves.

- `select=(None, 0, [0,1,2,3], 0), collapse='batch'`
  → one curve per global step, each = mean of the 4 batch trajectories.
- `select=(None, None, None, 0), collapse=['accum', 'batch']`
  → one curve per global step, each = mean over all accum × batch samples.

### Other features

- `normalize=True` divides each (post-collapse) curve by its value at `pos=-1`, so all curves meet at `(-1, 1.0)`.
- `window_avg=N` applies a centered rolling mean of width `N` along the token axis (1 = no smoothing).
- Provide `v_act` only / `v_grad` only / both. With both, act is dashed on the **left** y-axis and grad solid on the **right**, auto-aligned at `pos=-1`.
- **Color**: low index → yellow, high index → purple — *except* for `layer`, where low → purple, high → yellow. When several dims sweep, the *first* varying dim controls the direction.

### Recipes

- Single curve `(s=0, a=0, b=0, l=0)`:                  `select=(0, 0, 0, 0)`
- Sweep all layers at last step:                        `select=(-1, 0, 0, None)`
- Sweep all recorded steps:                             `select=(None, 0, 0, 0)`
- Sweep over batches:                                   `select=(-1, 0, None, 0)`
- Sweep both layers and steps:                          `select=(None, 0, 0, None)`
- Step-index range (first 5):                           `select=(slice(0, 5), 0, 0, 0)`
- Layer range:                                          `select=(-1, 0, 0, slice(0, 6))`
- Per-step mean over selected batches:                  `select=(None, 0, [0,1,2,3], 0), collapse='batch'`
- Per-step mean over accum × batch:                     `select=(None, None, None, 0), collapse=['accum', 'batch']`

In [2]:
DIM_NAMES          = ('step', 'accum', 'batch', 'layer')
DIM_AXES           = {'step': 0, 'accum': 1, 'batch': 2, 'layer': 3}
# Color direction per dim. True  -> low index = yellow, high index = purple
#                          False -> low index = purple, high index = yellow.
DIM_COLOR_REVERSED = {'step': True, 'accum': True, 'batch': True, 'layer': False}


def plot_norms(
    v_act:     Optional[View],
    v_grad:    Optional[View],
    select:    Tuple[Union[int, list, slice, None], ...],
    collapse:  Union[None, str, List[str]] = None,
    normalize: bool = False,
    window_avg: int = 1,
    title:     str = '',
):
    """Plot per-token norms over (step, accum, batch, layer).

    See the markdown cell above for the full API. Quick notes:
      * `select` is a 4-tuple `(step, accum, batch, layer)`. Each entry is
        an int (fix), list[int] (sweep), slice / np.s_[a:b] (sweep range),
        or None (sweep all).
      * `collapse` is None, a dim name, or a list of dim names from
        {'step', 'accum', 'batch', 'layer'}. Indices selected on those dims
        are *averaged* instead of plotted as separate curves.
      * `normalize` divides each (post-collapse) curve by its pos=-1 value.
      * `window_avg` is a centered rolling mean along the token axis.

    The `step` entry indexes axis 0 (i.e. into `view.global_steps`), so
    sparse step recording is handled transparently.
    """
    assert v_act is not None or v_grad is not None, 'need at least one of v_act, v_grad'
    v_ref = v_act if v_act is not None else v_grad
    assert len(select) == 4, 'select must be a 4-tuple over (step, accum, batch, layer)'

    # parse collapse
    if collapse is None:
        collapse_list = []
    elif isinstance(collapse, str):
        collapse_list = [collapse]
    else:
        collapse_list = list(collapse)
    bad = set(collapse_list) - set(DIM_NAMES)
    if bad:
        raise ValueError(f'unknown collapse dim(s): {sorted(bad)}; valid: {DIM_NAMES}')
    collapse_set = set(collapse_list)

    S, A, B, L, T = v_ref.arr.shape
    universes = [S, A, B, L]

    def _expand(sel, n):
        if sel is None:                                 return list(range(n))
        if isinstance(sel, slice):                      return list(range(n))[sel]
        if isinstance(sel, (list, tuple, np.ndarray)):  return [int(x) % n for x in sel]
        return [int(sel) % n]

    idx_lists = [_expand(s, n) for s, n in zip(select, universes)]
    axes_to_avg = tuple(DIM_AXES[name] for name in DIM_NAMES if name in collapse_set)

    remaining_names     = [n for n in DIM_NAMES if n not in collapse_set]
    remaining_idx_lists = [idx_lists[DIM_AXES[n]] for n in remaining_names]
    combos_meta         = list(itertools.product(*remaining_idx_lists))
    varies              = {n: len(idx_lists[DIM_AXES[n]]) > 1 for n in remaining_names}

    def _curves(view):
        if view is None:
            return [None] * len(combos_meta)
        sub = view.arr[np.ix_(*idx_lists)]                 # (n_step, n_accum, n_batch, n_layer, T)
        if axes_to_avg:
            sub = sub.mean(axis=axes_to_avg)               # collapse averaged dims
        out = []
        for pos_tuple in itertools.product(*[range(d) for d in sub.shape[:-1]]):
            norms = _smooth(sub[pos_tuple].astype(np.float32), window_avg)
            if normalize:
                ref = float(norms[-1])
                if ref == 0:
                    out.append(None); continue
                norms = norms / ref
            out.append(norms)
        return out

    act_curves  = _curves(v_act)
    grad_curves = _curves(v_grad)

    records = []
    for meta, ac, gc in zip(combos_meta, act_curves, grad_curves):
        if v_act  is not None and ac is None: continue
        if v_grad is not None and gc is None: continue
        records.append((meta, ac, gc))
    if not records:
        raise ValueError('No valid data for selection.')

    both = v_act is not None and v_grad is not None

    if both:
        ymin_a, ymax_a = _global_padded([r[1] for r in records])
        ymin_g, ymax_g = _global_padded([r[2] for r in records])
        anchor_a = 1.0 if normalize else float(np.median([r[1][-1] for r in records]))
        anchor_g = 1.0 if normalize else float(np.median([r[2][-1] for r in records]))
        fa = (anchor_a - ymin_a) / max(ymax_a - ymin_a, 1e-30)
        fg = (anchor_g - ymin_g) / max(ymax_g - ymin_g, 1e-30)
        tgt = max(0.05, min(0.95, (fa + fg) / 2))
        ymin_a, ymax_a = _aligned_range(anchor_a, ymin_a, ymax_a, tgt)
        ymin_g, ymax_g = _aligned_range(anchor_g, ymin_g, ymax_g, tgt)
    else:
        col = 1 if v_act is not None else 2
        ymin_1, ymax_1 = _global_padded([r[col] for r in records])

    n = len(records)
    primary_var = next((name for name in remaining_names if varies[name]), None)
    reverse     = DIM_COLOR_REVERSED.get(primary_var, True)
    fracs       = [(1.0 - i / max(n - 1, 1)) if reverse else (i / max(n - 1, 1))
                   for i in range(n)]
    colors      = pc.sample_colorscale('Plasma', fracs) if n > 1 else ['#1f77b4']

    pos = v_ref.rel_positions

    def _label(meta):
        parts = []
        for name, val in zip(remaining_names, meta):
            if not varies[name]: continue
            if name == 'step':
                parts.append(f'step={int(v_ref.global_steps[val])}')
            else:
                parts.append(f'{name}={val}')
        return ', '.join(parts) or 'curve'

    fig = go.Figure()
    for i, (meta, ac, gc) in enumerate(records):
        color = colors[i]
        lbl   = _label(meta)
        if ac is not None:
            fig.add_trace(go.Scatter(
                x=pos, y=ac, mode='lines',
                name=(f'act  {lbl}' if both else lbl),
                yaxis='y1', legendgroup=lbl,
                line=dict(color=color, width=1.2, dash='dash'),
                hovertemplate=f'pos: %{{x}}<br>act: %{{y:.4f}} [{lbl}]<extra></extra>',
            ))
        if gc is not None:
            fig.add_trace(go.Scatter(
                x=pos, y=gc, mode='lines',
                name=(f'grad {lbl}' if both else lbl),
                yaxis=('y2' if both else 'y1'), legendgroup=lbl,
                line=dict(color=color, width=2.0),
                hovertemplate=f'pos: %{{x}}<br>grad: %{{y:.4f}} [{lbl}]<extra></extra>',
            ))

    norm_suffix = ' / pos=-1' if normalize else ''
    if both:
        ya_title, yg_title = f'Act norm{norm_suffix}', f'Grad norm{norm_suffix}'
        ya_range, yg_range = [ymin_a, ymax_a], [ymin_g, ymax_g]
    elif v_act is not None:
        ya_title, ya_range = f'Act norm{norm_suffix}',  [ymin_1, ymax_1]
    else:
        ya_title, ya_range = f'Grad norm{norm_suffix}', [ymin_1, ymax_1]

    tags = []
    if collapse_set:    tags.append(f"mean over {','.join(n for n in DIM_NAMES if n in collapse_set)}")
    if normalize:       tags.append('normalized to pos=-1')
    if window_avg > 1:  tags.append(f'window_avg={window_avg}')
    tag_str = f"  ({', '.join(tags)})" if tags else ''

    layout = dict(
        title=f'{title}{tag_str}' if title else tag_str.lstrip(),
        xaxis=dict(title='Relative token position  (-1 = last token)',
                   tickmode='array', tickvals=_tickvals(int(pos.min()))),
        yaxis=dict(title=ya_title, range=ya_range),
        legend=dict(x=1.02, y=1.0, xanchor='left', yanchor='top'),
        height=460, margin=dict(l=60, r=200, t=60, b=60),
    )
    if both:
        layout['yaxis2'] = dict(title=yg_title, range=yg_range,
                                overlaying='y', side='right')
    fig.update_layout(**layout)
    return fig


## Config

In [4]:
NORMS_DIR  = '/home/svu/xudong_shen/myscratch/nanochat/logs/d12_ctx2048_04251714/norms'
STEP_START = 0      # inclusive lower bound on global_step
STEP_END   = 1660    # inclusive upper bound on global_step


## Load data

`read_hidden` returns `(S, A, R, B, L, T)` arrays plus the sparse
`global_steps` vector. We flatten `(R, B)` -> `batch_idx` so plot
indexing is 4-D `(global_step, accum_step, batch_idx, layer)` ahead of
the always-implicit `token_pos` axis.

In [5]:
h = read_hidden(NORMS_DIR, STEP_START, STEP_END)

S, A, R, B_per_rank, L, T = h.act.shape
act_flat  = h.act.reshape(S, A, R * B_per_rank, L, T)
grad_flat = h.grad.reshape(S, A, R * B_per_rank, L, T)

rel_positions = np.arange(T, dtype=np.int64) - T   # last token at -1

v_act  = View(act_flat,  h.global_steps, rel_positions)
v_grad = View(grad_flat, h.global_steps, rel_positions)

print(f'Recorded global_steps ({S}): {h.global_steps.tolist()}')
print(f'Shape (S, A, B, L, T) = {v_act.arr.shape}')
print(f'  S = {S} global_steps,  A = {A} accum_steps,  B = {R * B_per_rank} batch_idx '
      f'(rank {R} x device_batch {B_per_rank}),  L = {L} layers,  T = {T} tokens')
print(f'Layer types: {h.layer_types}')
print(f'act bytes: {act_flat.nbytes/1e6:.1f} MB   grad bytes: {grad_flat.nbytes/1e6:.1f} MB')


Recorded global_steps (84): [0, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400, 420, 440, 460, 480, 500, 520, 540, 560, 580, 600, 620, 640, 660, 680, 700, 720, 740, 760, 780, 800, 820, 840, 860, 880, 900, 920, 940, 960, 980, 1000, 1020, 1040, 1060, 1080, 1100, 1120, 1140, 1160, 1180, 1200, 1220, 1240, 1260, 1280, 1300, 1320, 1340, 1360, 1380, 1400, 1420, 1440, 1460, 1480, 1500, 1520, 1540, 1560, 1580, 1600, 1620, 1640, 1660]
Shape (S, A, B, L, T) = (84, 16, 16, 12, 2048)
  S = 84 global_steps,  A = 16 accum_steps,  B = 16 batch_idx (rank 1 x device_batch 16),  L = 12 layers,  T = 2048 tokens
Layer types: ['full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention', 'full_attention']
act bytes: 2113.9 MB   grad bytes: 2113.9 MB


## Demos
Each cell is a single `plot_norms` call — vary `select`, `normalize`,
`window_avg` to taste.

### Single trajectory: act only

In [6]:
plot_norms(v_act, None,
           select=(1660, 0, 0, 0),
           window_avg=200,
           title='Hidden activation norm').show()


### Single trajectory: grad only

In [7]:
plot_norms(None, v_grad,
           select=(1660, 0, 0, 0),
           window_avg=200,
           title='Hidden gradient norm').show()


### Act + grad overlay (raw)

In [8]:
plot_norms(v_act, v_grad,
           select=(1660, 0, 0, 0),
           window_avg=200,
           title='Act & grad norm').show()


### Act + grad overlay (normalized to pos=-1)

In [13]:
plot_norms(v_act, v_grad,
           select=(1660, 0, 0, 0),
           normalize=True, window_avg=200,
           title='Act & grad norm').show()


### Sweep over recorded steps (sparse-safe)
Legend labels show the **actual** `global_step` ids from
`view.global_steps`, not the axis-0 index. Step-only sweep uses the
*reversed* Plasma direction: step=0 → yellow, last step → purple.

In [15]:
plot_norms(v_act, None,
           select=(slice(0,10), 0, None, 0),
           normalize=True, window_avg=200,
           title='Act norm — all recorded steps').show()

plot_norms(None, v_grad,
           select=(slice(0,10), 0, None, 0),
           normalize=True, window_avg=200,
           title='Grad norm — all recorded steps').show()


### Sweep over layers
Layer-only sweep uses the **non-reversed** Plasma direction:
layer 0 → purple, last layer → yellow.

In [17]:
plot_norms(v_act, None,
           select=(1660, 0, 0, None),
           normalize=True, window_avg=200,
           title='Act norm — all layers').show()

plot_norms(None, v_grad,
           select=(1660, 0, 0, None),
           normalize=True, window_avg=200,
           title='Grad norm — all layers').show()


### Collapse: per-step mean over batches
One curve per global_step, each = mean of `batch_idx ∈ [0,1,2,3]`.

In [20]:
plot_norms(None, v_grad,
           select=(None, 0, None, 0),
           collapse='batch',
           normalize=True, window_avg=1,
           title='Grad norm — per-step mean over batches').show()


### Collapse: per-step mean over accum × batch
One curve per global_step, averaging out both gradient-accumulation
micro-steps **and** all batch trajectories.

In [21]:
plot_norms(None, v_grad,
           select=(None, None, None, 0),
           collapse=['accum', 'batch'],
           normalize=True, window_avg=200,
           title='Grad norm — per-step mean over accum x batch').show()
